# Overview 

## Client side
```python
client = MultiServerMCPClient (
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": 
                ["/path/to/script.py"],
        },
        "weather": {
            "transport": "http",
            "url": "http://localhost:8000/mcp
        }
    }
)

tools = client.get_tools ()

agent = create_agent (
    model="...",
    tools,
    ...
)

result1 = await agent.ainvoke (
    {"messages": [{"role": "user", "content":"What is the result of 2*3"}]}
)

result2 = await agent.ainvoke (
    "What is the wheather in CA"
)
```

## Server side (custom servers)

Test the above client code by launching local custom servers through the FastMCP library

```python

mcp = FastMCP("Math")

@mcp.tool
async def plus (a: int, b: int):
    return a+b

if __name__ == "__main__":
    mcp.run(transport="stdio")

```

```python

mcp = FastMCP("Weather")

@mcp.tool
async def get_weather (city: str):
    return "sunny"

if __name__ == "__main__":
    mcp.run(transport="http", 
        # url="localhost",
        # port=8000
    )

```

# Transports

## Stateless vs Statefull

- By default it is stateless: new server started at each call.

## HTTP

You can:
- Indicate headers
- Authenticate

## stdio

You can:

- communicate via strings as in command line input / output

## Statefull

- Use statefull with sessions
- For example if the server maintains context across tool invocations

```python
async with client.session ():
    tools = await load_mcp_tools ()
    agent = create_agent (
        model="...",
        tools,
    )
```

## stateful functionalities: 

We can:
- Get the tools
- Get structured content:
    - using the `artifact` field of `ToolMessage`
    - using `interceptors` e.g., to automatically append the structured content to tool result:

```python
async def my_interceptor (request: MCPToolCallRequest, handler):
    result = await handler (request)
    # result.structuredContent: dict

client = MultiServerMCPClient ({...}, tool_interceptors=[my_interceptor])
```

## Get multimedia content

```python
response = await agent.ainvoke...
response["messages"]
for message in response["messages"]:
    for block in message.content_blocks:
        if block["type"] == ... # "text", "image", ...
            block.get(...)
```

## Get resources 

```python
blobs = await client.get_resources("server_name", uris=...)
for blob in blobs:
    ...
```

## get prompts

```python
messages = await client.get_prompt (...)
```

## interceptors: accessing runtime context

- Access user details (ID, api keys, ...)
```python

class ContextSchema:
    my_field: ...

async def my_interceptor (request: MCPToolCallRequest, handler):
    result = await handler (request)    
    # request.runtime.context.my_field
    modified_request = request.override (
        args = {
            **request.args,
            {"my_field": my_field}
        }
    )
    return await handler (modified_request)

client = MultiServerMCPClient ({...}, tool_interceptors=[my_interceptor])

tools = client.get_tools (...)

agent = create_agent (
    model="",
    tools,
    context_scheme=ContextSchema,
)

result = await agent.ainvoke (
    "...",
    context={"my_field": my_value},
)
```


## Store and State

- Similar to the above, see context engineering notes

## Tool Call ID

Allows to track tool executions, return properly formatted responses, etc.

```python
...
tool_call_id = request.runtime.tool_call_id
if there_is_issue(request.name):
    return ToolMessage (
        content="There is X issue (e.g., rate limit exceeded)",
        tool_call_id=tool_call_id,
    )

result = await handler(request)
log_tool_execution (tool_call_id, request.name, success=True)
...
```

## State updates and commands

Interceptors can return `Command` objects to change the state or the flow of the execution.

```python
async def my_interceptor (
    ...
) -> Command:
    result = await handler(request)
    if condition:
        return Command (
            update={
                "messages": [result] if isinstance (result, ToolMessage) else [],
                "task_status": "completed",
            },
            goto = "my_next_step" # can be "__end__"
        )
    return result
```

## Custom interceptors

You can:
- Modify the request before calling the tool 
    - use `request.override(...)` before `result = handler(request)`
    - For instance, change args or header, see above
- Perform the action after calling the tool, e.g., manage exceptions
```python
try:
    result = await handler(request)
except Exception as e:
    print (request.name)
    ...
```

## Compose interceptors

```python
async def outer (...):
    print ("outer before")
    result = await handler (request)
    print ("outer after")
    return result

async def inner (...):
    print ("inner before")
    result = await handler (request)
    print ("inner after")
    return result
```

- Onion structure:
```python

client = MultiServerMCPClient (
    {...},
    tool_interceptors = [outer, inner]
)
# Execution: outer before -> inner before -> tool -> inner after -> outer after

```    

## Callbacks

```python
async def my_built_in_callback (
    callback_specific_1,
    ...
    context: CallbackContext,
):
    # context.my_built_in_callback_specific_field

client = MultiServerMCPClient (
    {...},
    callbacks=Callbacks (my_built_in_callback=my_built_in_callback)
)
```

where `my_callback` can be one of the built-in callbacks: 
    - on_progress
    - on_logging_message
    - ...

## Elicitation

Allows to interactively obtain information from user from tool

- Use ctx.elicit() in server's tool implementation, using schema

```python
class MySchema
...
@mcp.tool ()
async def my_tool (name, ctx: Context):
    result = await ctx.elicit (
        message="...",
        schema=MySchema
    )
    if result.action=="accept" and result.data:
        # result.data.my_field_1, ...
    # result.action can also be "decline" or "cancel"
```

- Use `on_elicitation` callback in client code:
```python
async def on_elicitation (
    ... # see documentation
):
    return ElicitResult (
        action="accept",
        content={"my_field_1": my_value_1, ...}
    )
    ...
```